# Representation learning & recommender systems

## Part A: matrix factorization, from vanilla autograd to `nn.Module`

Vincent Guigue (vincent.guigue@agroparistech.fr)

In this practical session, we investigate two classical matrix-factorization models and their neural network implementation.

**Important note:** this tutorial is dedicated to beginners. We start with basic `pytorch` structures only (tensors + autograd, no `nn.Module`, no optimizer), then we move step by step to the standard chain (`nn.Embedding`, `nn.Module`, `DataLoader`, optimizer). The following notebooks are directly built around these standard modules.

**Roadmap:** baseline with 3 scalars per prediction $\to$ embeddings $\to$ classic matrix factorization $\to$ Koren's model (factorization + biases).

## Table of contents

- [0. Configuration (device, tensorboard)](#sec-1)
- [1. Data: MovieLens 100k](#sec-2)
- [2. Baseline model with pytorch's vanilla autograd](#sec-3)
  - [2.1 Parameters](#sec-4)
  - [2.2 Prediction and error functions](#sec-5)
  - [2.3 Evaluation, without any optimization for now](#sec-6)
  - [2.4 Optimizing the parameters with (stochastic) gradient descent](#sec-7)
- [3. The Embedding module](#sec-8)
  - [3.1 Main difficulty = dealing with batches!](#sec-9)
  - [3.2 \[Python tuto\] Dealing with an undefined number of values](#sec-10)
- [4. Classic matrix factorization (called SVD in RecSys)](#sec-11)
  - [4.1 Model definition](#sec-12)
  - [4.2 Data pipeline: `TensorDataset` and `DataLoader`](#sec-13)
  - [4.3 Loss function](#sec-14)
  - [4.4 Training loop](#sec-15)
- [5. (Your turn) Koren 2009 model](#sec-16)
- [6. \[Optional\] How to complete this series of experiments](#sec-17)
  - [6.1 Visualization of the item embeddings](#sec-18)
  - [6.2 Regularization with side information](#sec-19)

# <a id="sec-1"></a>0. Configuration (device, tensorboard)

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Device selection: NVIDIA GPU (cuda) > Apple GPU (mps) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
# NB: the models of this notebook are tiny: the CPU is often faster than mps (~3x on a Mac M1)
# device = torch.device("cpu")
print("device:", device)

torch.manual_seed(0)
rng = np.random.default_rng(0)

device: cuda


In [ ]:
from torch.utils.tensorboard import SummaryWriter

import tempfile
TB_PATH = Path(tempfile.gettempdir()) / "logs" / "module3"
TB_PATH.mkdir(parents=True, exist_ok=True)
TB_INLINE = True   # True: tensorboard displayed in the notebook / False: launch it yourself in a terminal
print(f"To launch tensorboard in a terminal: tensorboard --logdir {TB_PATH}")
if TB_INLINE:
    %load_ext tensorboard
    %tensorboard --logdir {TB_PATH}b

To launch tensorboard in a terminal: tensorboard --logdir C:\Users\THEOPO~1\AppData\Local\Temp\logs\module3
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


# <a id="sec-2"></a>1. Data: [MovieLens 100k](https://grouplens.org/datasets/movielens/)

A very common dataset describing users, movies & interactions (100,000 ratings from 1 to 5, given by 943 users on 1682 movies):

![image reco](media/Facto-mat.png)

To be able to embed the data easily, we need to remap the user/item ids into $[0, N_{users}[$ and $[0, N_{items}[$ (they will be used as row indexes in the embedding matrices).

In [4]:
## Load (we only use pandas to load the data... no pandas skills required for this session!)
ratings = pd.read_csv("data/ml-100k/u.data", sep="\t", names=["userId", "movieId", "rating", "timestamp"])
ratings["rating"] = ratings["rating"].astype(float)
print(ratings.shape)
ratings.head(5)

(100000, 4)


,userId,movieId,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [5]:
## Remap the ids into [0, N[
user_map = {user: num for num, user in enumerate(ratings["userId"].unique())}
item_map = {item: num for num, item in enumerate(ratings["movieId"].unique())}

num_users = len(user_map)
num_items = len(item_map)

ratings["userId"] = ratings["userId"].map(user_map)
ratings["movieId"] = ratings["movieId"].map(item_map)
print(f"{num_users} users, {num_items} items")
ratings.head(5)

943 users, 1682 items


,userId,movieId,rating,timestamp
0,0,0,3.0,881250949
1,1,1,3.0,891717742
2,2,2,1.0,878887116
3,3,3,2.0,880606923
4,4,4,1.0,886397596


**Train / validation / test split.** Random split: 64% train, 16% validation (to tune and compare models), 20% test (used **once**, at the very end, to report the final performance).

*Note:* a few users or items of the validation/test sets may have no rating in the train set: their parameters will never be updated (they stay at their initial value).

In [6]:
perm = rng.permutation(len(ratings))
n_test, n_val = len(ratings) // 5, len(ratings) * 16 // 100

test_ratings = ratings.iloc[perm[:n_test]]
val_ratings = ratings.iloc[perm[n_test:n_test + n_val]]
train_ratings = ratings.iloc[perm[n_test + n_val:]]

print(f"#train: {len(train_ratings)}, #val: {len(val_ratings)}, #test: {len(test_ratings)}")

#train: 64000, #val: 16000, #test: 20000


In [7]:
# USAGE: in what follows, we browse the ratings this way (itertuples yields: index, user, item, rating, timestamp)
for cpt, (index, uid, mid, r, ts) in enumerate(train_ratings.itertuples()):
    print(index, uid, mid, r)   # the indexes were shuffled by the split
    if cpt >= 5:
        break

33958 13 1180 3.0
22316 268 719 5.0
50853 635 156 5.0
79962 43 350 1.0
733 138 462 4.0
7585 194 433 3.0


# <a id="sec-3"></a>2. Baseline model with pytorch's vanilla autograd

Your goal now is to reproduce the following (strong) baseline model, from the [surprise](https://surprise.readthedocs.io/en/stable/basic_algorithms.html#surprise.prediction_algorithms.baseline_only.BaselineOnly) library:

$$\hat{r}_{ui} = b_{ui} = \mu + b_u + b_i, \qquad (\mu,b_u,b_i) \in \mathbb R^3$$

[no matrix factorization here, <font color="red">only 3 scalars</font> involved for a prediction $(u,i)$] <BR>
[$\mu$ could be computed from the train set (mean rating), but we are going to learn it in the optimization process]

## <a id="sec-4"></a>2.1 Parameters

All parameters are 1-dimensional tensors:
- **mu:** the global mean, shape `(1,)`
- **bu:** the user biases, shape `(num_users,)` $\Rightarrow$ `bu[u]` is the bias of user `u`
- **bi:** the item biases, shape `(num_items,)`

In [8]:
mu = torch.tensor([3.5], requires_grad=True)          # activate the gradient to be able to learn something
bu = torch.zeros(num_users, requires_grad=True)
bi = torch.zeros(num_items, requires_grad=True)

# # using directly the Embedding module (see section 3), it would give:
# mu = torch.nn.Embedding(1, 1)            # only one scalar to learn for the whole set
# bu = torch.nn.Embedding(num_users, 1)    # one scalar per user
# bi = torch.nn.Embedding(num_items, 1)    # one scalar per item

In [12]:
# TODO: check the dimensions of the created structures
print(mu.size(),
bu.size(),
bi.size())

torch.Size([1]) torch.Size([943]) torch.Size([1682])


## <a id="sec-5"></a>2.2 Prediction and error functions

Define two functions:
- `predict(u, i)`: returns the prediction $\hat{r}_{ui} = \mu + b_u + b_i$ for the pair (user, item)
- `error(pred, real)`: returns the squared error $(\hat r - r)^2$ (the mean over the dataset will give the MSE)

In [13]:
def predict(u, i):
    return mu + bu[u] + bi[i]

def error(pred, real):
    return (pred - real) ** 2

# validation on user 0 and item 0
print(predict(0, 0), error(predict(0, 0), 4.))

tensor([3.5000], grad_fn=<AddBackward0>) tensor([0.2500], grad_fn=<PowBackward0>)


## <a id="sec-6"></a>2.3 Evaluation, without any optimization for now

Write a function `evaluate(df)` that returns the MSE of the model over the ratings of the DataFrame `df` (browse it with `itertuples`, cf. above). No gradient is needed here: use `with torch.no_grad():`.

Then display the train, validation and test errors. Bad results expected (and close to each other, since nothing has been learned yet)... It is just to check that the function works.

In [20]:
# with torch.no_grad() => boucle for + error(predict)
def evaluate(df):
    with torch.no_grad():
        P, E = [], []
        for index, uid, mid, r, ts in df.itertuples():
            P.append(predict(uid, mid))
            E.append(error(P[-1], r).item())
        return np.mean(E)

print(f"MSE train: {evaluate(train_ratings):.3f} | val: {evaluate(val_ratings):.3f} | test: {evaluate(test_ratings):.3f}")

MSE train: 1.268 | val: 1.279 | test: 1.260


## <a id="sec-7"></a>2.4 Optimizing the parameters with (stochastic) gradient descent

For each epoch, loop over the training ratings (in a random order: `train_ratings.sample(frac=1)`) and, **for each rating**:
1. compute the prediction and the error,
2. backpropagate (`backward`),
3. update the parameters **without** tracking the operations (`with torch.no_grad():`): $\theta \leftarrow \theta - lr \cdot \nabla_\theta$,
4. reset the gradients to zero (`p.grad.zero_()`).

At the end of each epoch, display the train error (mean of the errors seen during the epoch) and the validation error, and log them in tensorboard (`writer.add_scalar`).

*Note:* `bu.grad` is a vector of size `num_users` that is zero except at position `u`: updating the whole vector is simple (and correct), but a bit wasteful. This is exactly what `nn.Embedding` with a sparse gradient avoids... we won't go that far.

In [21]:
lr = 0.01
n_epochs = 5
writer = SummaryWriter(TB_PATH / f"baseline-{time.strftime('%H%M%S')}")

for epoch in range(n_epochs):
    train_e = 0
    for index, uid, mid, r, ts in train_ratings.sample(frac=1).itertuples():
        se = error(predict(uid, mid), r)
        train_e += se.item()
        se.backward()
        with torch.no_grad():
            for p in [mu, bu, bi]:
                p -= lr * p.grad
                p.grad.zero_()   # manually zero the gradients after updating the weights
    train_e /= len(train_ratings)
    val_e = evaluate(val_ratings)
    writer.add_scalar("mse/train", train_e, epoch)
    writer.add_scalar("mse/val", val_e, epoch)
    print(f"epoch {epoch} | train MSE: {train_e:.3f} | val MSE: {val_e:.3f}")

writer.close()
mse_baseline = evaluate(test_ratings)
print(f"Final test MSE (baseline): {mse_baseline:.3f}")

epoch 0 | train MSE: 1.017 | val MSE: 0.942
epoch 1 | train MSE: 0.910 | val MSE: 0.938
epoch 2 | train MSE: 0.888 | val MSE: 0.923
epoch 3 | train MSE: 0.877 | val MSE: 0.899
epoch 4 | train MSE: 0.872 | val MSE: 0.914
Final test MSE (baseline): 0.905


# <a id="sec-8"></a>3. The Embedding module

To build a matrix of vectorial representations of dimension $Z$, for instance describing the users, we are going to use a new module called [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html):

$ U = \begin{pmatrix}\mathbf u_1, \ldots, \mathbf u_n\end{pmatrix}, \mathbf u \in \mathbb R^Z $

![image reco](media/Facto-mat.png)

Call it with an index (a tensor of integers), get a $Z$ dimensional representation:

In [22]:
latent_size = 10
nb_users = 100   # arbitrary
nb_items = 50
users = nn.Embedding(nb_users, latent_size)   # random init
items = nn.Embedding(nb_items, latent_size)   # random init

# get the representation of user 5:
print("User 5:", users(torch.tensor(5)))      # WARNING: call it with a tensor (not an int)

# get the representations of users 5 & 7:
print("User 5 & 7:", users(torch.tensor([5, 7])))

User 5: tensor([ 1.0554,  0.1778, -0.2303, -0.3918,  0.5433, -0.3952, -0.4462,  0.7440,
         1.5210,  3.4105], grad_fn=<EmbeddingBackward0>)
User 5 & 7: tensor([[ 1.0554,  0.1778, -0.2303, -0.3918,  0.5433, -0.3952, -0.4462,  0.7440,
          1.5210,  3.4105],
        [-0.4731,  0.3356, -1.6293, -0.5497, -0.4798, -0.4997, -1.0670,  1.1149,
         -0.1407,  0.8058]], grad_fn=<EmbeddingBackward0>)


In [23]:
# Initialize the embedding with smaller values:
nn.init.normal_(users.weight, 0, 0.01)    # apply on the weights (in place: function name ending with _)
print("User 5:", users(torch.tensor(5)))

User 5: tensor([-0.0005, -0.0064, -0.0047, -0.0105,  0.0043, -0.0155, -0.0099,  0.0070,
        -0.0050, -0.0148], grad_fn=<EmbeddingBackward0>)


## <a id="sec-9"></a>3.1 Main difficulty = dealing with batches!

Based on a very simple matrix factorization formulation:
$$ \hat r = \mathbf v_i^T \mathbf v_u ,\qquad \mathbf v_i, \mathbf v_u \in \mathbb R^z$$
Are you able to compute $\hat r$ for a batch of indexes `ind`? The expected result is a vector of size `len(ind)` (one score per pair), **not** a `len(ind) x len(ind)` matrix.

In [25]:
# batch of indexes
ind = torch.tensor([1, 2, 3])
u = users(ind)
i = items(ind)    # => check the size
print(ind.shape, u.shape, i.shape)

# compute i.T u for all pairs (u[k], i[k])
# idea to save useless computations:
# 1. pairwise (element-wise) multiplication
# 2. find the right sum to get correct dimensions (and probably correct results)
##  TODO 
r = torch.sum(u * i, dim=1)
print(r.shape, r)

torch.Size([3]) torch.Size([3, 10]) torch.Size([3, 10])
torch.Size([3]) tensor([0.0143, 0.0362, 0.0060], grad_fn=<SumBackward1>)


## <a id="sec-10"></a>3.2 <span style="color: blue;">[Python tuto]</span> Dealing with an undefined number of values

Widely used in python... And in particular in the next steps (the models will return a variable number of tensors to regularize).

In [27]:
# 1. a function can return many things (in a tuple)
def fonction():
    return 1, 2, 3, 4

a, b, c, d = fonction()    # default behavior => unpack the tuple
print(a, b, c, d)

# 2. you can gather the end in a list (here: all values but the first one)
_, *res = fonction()
print(res)

# 3. a function that takes an arbitrary number of parameters:
def fonction2(*params):
    for p in params:
        print(p)

fonction2(*res)    # unpack the list: equivalent to fonction2(2, 3, 4)
fonction2(res)     # without the star: a single parameter (the list)

1 2 3 4
[2, 3, 4]
2
3
4
[2, 3, 4]


# <a id="sec-11"></a>4. Classic matrix factorization (called SVD in RecSys)

We now implement a simple factorization model, with a global mean:
$$ \min\limits_{U,I}\sum\limits_{(u,i)} \underbrace{(r_{ui} -  (I_i^TU_u + \mu))^2}_\text{error} + \underbrace{\lambda(\|U_u\|^2+\|I_i\|^2) }_\text{regularization} $$

where the prediction is done in the following way:
$$\hat r_{ui} = \mu + U_u^T I_i $$

where $\mu$ is the global mean, $U_u$ a user embedding and $I_i$ an item embedding.

**Steps** to implement such a model in pytorch:
 1. model definition
 2. data pipeline (`Dataset` + `DataLoader`)
 3. loss function
 4. training/evaluation loop

## <a id="sec-12"></a>4.1 Model definition

A model class typically extends [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html), the neural network module. It is a convenient way of encapsulating parameters, with helpers for moving them to GPU (`.to(device)`), exporting, loading, etc.

Two methods have to be defined:
- `__init__`: initializes the model parameters (here: the user embeddings $U$, the item embeddings $I$, the global mean $\mu$),
- `forward`: the transformation from input to output. When doing `model(input)`, you actually call this method.

`forward(user, item)` receives **two tensors of indexes** (a batch of users and the corresponding batch of items) and returns:
- the predictions (a vector of size `batch`): $\hat r_{ui} = \mu + U_u^T I_i$, i.e. `torch.sum(embed_u * embed_i, -1) + self.mean` (cf. section 3.1),
- the embeddings `embed_u`, `embed_i` of the batch, to regularize them in the loss.

**TODO:** complete the following `__init__` and `forward` methods.

In [32]:
class ClassicMF(nn.Module):

    def __init__(self, nb_users, nb_items, latent_size):
        super().__init__()
        self.name = "ClassicMF"
        # The global mean: a learned scalar (nn.Parameter = tensor registered as a parameter of the module)
        self.mean = nn.Parameter(torch.tensor(3.5))
        # Create the two embedding matrices: self.users and self.items
        # and initialize them with small values, e.g.: nn.init.normal_(self.users.weight, 0, 0.01)
        ##  TODO 
        self.users = nn.Embedding(nb_users, latent_size)
        self.items = nn.Embedding(nb_items, latent_size)
        nn.init.normal_(self.users.weight, 0, 0.01)
        nn.init.normal(self.items.weight, 0, 0.01)

    def forward(self, user, item):
        ##  TODO 
        embed_u, embed_i = self.users(user), self.items(item)
        out = torch.sum(embed_u * embed_i, -1) + self.mean
        return out, embed_u, embed_i   # prediction + embeddings to regularize

## <a id="sec-13"></a>4.2 Data pipeline: `TensorDataset` and `DataLoader`

The training loop is organized around the [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) class, which samples examples from a dataset and gathers them into mini-batches (here, 3 tensors: users, items, ratings).

Our data are already numbers: we simply store them into tensors and wrap them in a [`TensorDataset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset) (`dataset[k]` returns the triplet `(user_k, item_k, rating_k)`).

*Note:* when the examples cannot be directly stacked into tensors (e.g. texts of different lengths), a `collate_fn` function is given to the `DataLoader` to build the batches: see notebook 3_3.

In [29]:
def to_dataset(df):
    return TensorDataset(torch.tensor(df["userId"].values),
                         torch.tensor(df["movieId"].values),
                         torch.tensor(df["rating"].values, dtype=torch.float32))

batch_size = 256
train_loader = DataLoader(to_dataset(train_ratings), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(to_dataset(val_ratings), batch_size=1024)     # no need to shuffle for evaluation
test_loader = DataLoader(to_dataset(test_ratings), batch_size=1024)

users_t, items_t, ratings_t = next(iter(train_loader))   # first batch
print(users_t.shape, items_t.shape, ratings_t.shape, ratings_t.dtype)

torch.Size([256]) torch.Size([256]) torch.Size([256]) torch.float32


## <a id="sec-14"></a>4.3 Loss function

MSE + L2 regularization of the parameters involved in the predictions of the batch. Thanks to the `*params` syntax (cf. 3.2), the same function will work whatever the number of tensors returned by the model.

In [30]:
def loss_func(pred, ratings, reg, *params):
    """MSE + reg * (mean over the batch of the squared norms of each parameter tensor of shape (batch, dim))
    returns: (loss to optimize, mse)"""
    mse = F.mse_loss(pred, ratings)
    l2 = sum(p.pow(2).sum(-1).mean() for p in params)
    return mse + reg * l2, mse

In [33]:
## INTERMEDIATE BOX for in depth understanding
model = ClassicMF(num_users, num_items, 25)

# forward on the first batch (if your forward is defined as expected)
pred, embed_u, embed_i = model(users_t, items_t)
print(pred.shape, embed_u.shape)     # (batch,) (batch, latent_size)

# alternative syntax: gather all the parameters to regularize in a list...
pred, *params = model(users_t, items_t)
print(len(params), params[0].shape)  # params[0] corresponds to embed_u

# ... and give the list to loss_func, unpacked
print(loss_func(pred, ratings_t, 0.1, *params))

torch.Size([256]) torch.Size([256, 25])
2 torch.Size([256, 25])
(tensor(1.3208, grad_fn=<AddBackward0>), tensor(1.3203, grad_fn=<MseLossBackward0>))


C:\Users\Theo portable\AppData\Local\Temp\ipykernel_11272\1781116679.py:14: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  nn.init.normal(self.items.weight, 0, 0.01)


## <a id="sec-15"></a>4.4 Training loop

Complete the two functions:
- `evaluate_mf(model, loader)`: MSE of the model on all the examples of `loader` (beware: the last batch can be smaller, compute a **sum** of squared errors then divide by the number of examples),
- `train_mf(model, n_epochs, lr, reg)`: for each epoch, loop over `train_loader` (zero_grad, forward, loss, backward, step), then compute the validation MSE; log the train and validation MSE in tensorboard and print them.

Don't forget to send the model and the batches to `device`, and to switch the model between `model.train()` and `model.eval()` (useless here, but good practice: it matters as soon as there is dropout or batch-norm).

In [34]:
def evaluate_mf(model, loader):
    model.eval()
    se, n = 0., 0
    with torch.no_grad():
        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            pred, *_ = model(u, i)
            se += F.mse_loss(pred, r, reduction="sum").item()
            n += len(r)
    return se / n
    


def train_mf(model, n_epochs=8, lr=1e-2, reg=0.1):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    writer = SummaryWriter(TB_PATH / f"{model.name}-{time.strftime('%H%M%S')}")
    for epoch in range(n_epochs):
        model.train()
        cum_mse, n = 0., 0
        for u, i, r in train_loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            optimizer.zero_grad()
            pred, *params = model(u, i)
            loss, mse = loss_func(pred, r, reg, *params)
            loss.backward()
            optimizer.step()
            cum_mse += mse.item() * len(r)
            n += len(r)
        val_mse = evaluate_mf(model, val_loader)
        writer.add_scalar("mse/train", cum_mse / n, epoch)
        writer.add_scalar("mse/val", val_mse, epoch)
        print(f"[{model.name}] epoch {epoch} | train MSE: {cum_mse / n:.3f} | val MSE: {val_mse:.3f}")
    writer.close()
    return model

In [35]:
model_mf = train_mf(ClassicMF(num_users, num_items, 25))
mse_mf = evaluate_mf(model_mf, test_loader)
print(f"Final test MSE (ClassicMF): {mse_mf:.3f}")

C:\Users\Theo portable\AppData\Local\Temp\ipykernel_11272\1781116679.py:14: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  nn.init.normal(self.items.weight, 0, 0.01)


[ClassicMF] epoch 0 | train MSE: 1.152 | val MSE: 1.008
[ClassicMF] epoch 1 | train MSE: 0.892 | val MSE: 0.932
[ClassicMF] epoch 2 | train MSE: 0.778 | val MSE: 0.910


KeyboardInterrupt: 

**Question.** Compare the train and validation curves (in tensorboard): what do you observe? Try a weaker regularization (`reg=0.01`). How could we choose the number of epochs automatically?

In [ ]:
##  TODO 

# <a id="sec-16"></a>5. (Your turn) Koren 2009 model

This model simply adds a bias for each user and for each item (it combines the baseline of section 2 and the factorization of section 4):

$$ \min\limits_{U,I,b}\sum\limits_{(u,i)} \underbrace{(r_{ui} -  (I_i^TU_u + \mu+ b_i+b_u))^2}_\text{error} + \underbrace{\lambda(\|U_u\|^2+\|I_i\|^2 + b_u^2 + b_i^2) }_\text{regularization} $$

$$\hat r_{ui} = \mu + b_u + b_i + U_u^T I_i $$

**TODO:**
- (a) complete the model initialization: the biases $b_u$, $b_i$ can be stored in `nn.Embedding(n, 1)` (one scalar per user/item), **initialized to 0**,
- (b) complete the forward method: it returns the prediction, then all the tensors to regularize (`embed_u, embed_i, bias_u, bias_i`).

Beware of the shapes: `self.ubias(user)` has shape `(batch, 1)`, the prediction must have shape `(batch,)` (use `.squeeze(-1)`). Keep the biases with shape `(batch, 1)` in the returned values: `loss_func` sums over the last dimension.

In [ ]:
##  TODO 

The training loop stays the same (the loss function adapts to the number of returned tensors), you only have to change the model:

In [24]:
model_koren = train_mf(KorenMF(num_users, num_items, 25))
mse_koren = evaluate_mf(model_koren, test_loader)

print(f"\nTest MSE | baseline: {mse_baseline:.3f} | ClassicMF: {mse_mf:.3f} | KorenMF: {mse_koren:.3f}")

[KorenMF] epoch 0 | train MSE: 1.027 | val MSE: 0.912


[KorenMF] epoch 1 | train MSE: 0.822 | val MSE: 0.870


[KorenMF] epoch 2 | train MSE: 0.729 | val MSE: 0.865


[KorenMF] epoch 3 | train MSE: 0.660 | val MSE: 0.863


[KorenMF] epoch 4 | train MSE: 0.611 | val MSE: 0.864


[KorenMF] epoch 5 | train MSE: 0.580 | val MSE: 0.867


[KorenMF] epoch 6 | train MSE: 0.558 | val MSE: 0.868


[KorenMF] epoch 7 | train MSE: 0.543 | val MSE: 0.868



Test MSE | baseline: 0.891 | ClassicMF: 0.922 | KorenMF: 0.867


**Question.** Rank the 3 models. Why does the classic factorization do worse than the 3-scalar baseline?

In [ ]:
##  TODO 

# <a id="sec-17"></a>6. [Optional] How to complete this series of experiments

## <a id="sec-18"></a>6.1 Visualization of the item embeddings

The [t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) algorithm projects the embeddings in 2D. One of its best implementations is available in tensorboard: run the cell below, then open the **Projector** tab of tensorboard (reload the page if needed) and select t-SNE. Look for movies that you know: are their neighbors similar?

In [26]:
# movie titles (u.item: id | title | ...), in the order of our internal indexes
items_info = pd.read_csv("data/ml-100k/u.item", sep="|", encoding="latin-1", header=None, usecols=[0, 1], names=["movieId", "title"])
idx2title = dict(zip(items_info["movieId"].map(item_map), items_info["title"]))
titles = [idx2title[k] for k in range(num_items)]

writer = SummaryWriter(TB_PATH / f"projector-{time.strftime('%H%M%S')}")
writer.add_embedding(model_koren.items.weight.detach().cpu(), metadata=titles, tag="items_koren")
writer.close()

# same idea directly in the notebook: nearest neighbors (cosine similarity) of a movie
def neighbors(emb, k_item, k=5):
    E = F.normalize(emb.detach().cpu(), dim=1)
    sims = E @ E[k_item]
    return [titles[j] for j in sims.topk(k + 1).indices[1:]]

star_wars = titles.index("Star Wars (1977)")
print(neighbors(model_koren.items.weight, star_wars))

['Return of the Jedi (1983)', 'Empire Strikes Back, The (1980)', 'Giant (1956)', 'Of Human Bondage (1934)', 'Raiders of the Lost Ark (1981)']


## <a id="sec-19"></a>6.2 Regularization with side information

Exploit side information to regularize the profiles: users from the same age category are supposed to have closer representations, movies from the same genre, etc. (see notebook 3_2 for an example with a triplet loss).

In [27]:
# load side information about the users
uinfo = pd.read_csv("data/ml-100k/u.user", sep="|", names=["userId", "age", "gender", "occupation", "zip"])

# WARNING: we changed the definition of the ids => make the ids consistent (using the same dictionary)
uinfo["userId"] = uinfo["userId"].map(user_map)
gender_map = {g: num for num, g in enumerate(uinfo["gender"].unique())}
uinfo["gender"] = uinfo["gender"].map(gender_map)
occupation_map = {p: num for num, p in enumerate(uinfo["occupation"].unique())}
uinfo["occupation"] = uinfo["occupation"].map(occupation_map)
uinfo["age_cat"] = pd.cut(uinfo["age"], bins=[0, 18, 25, 35, 45, 55, 100], labels=False)   # age categories
uinfo.head(5)

,userId,age,gender,occupation,zip,age_cat
0,118,24,0,0,85711,1
1,229,53,1,1,94043,4
2,269,23,0,2,32067,1
3,267,24,0,0,43537,1
4,111,33,1,1,15213,2
